# CPUC Autonomous Vehicle Data - Exploratory Analysis

This notebook provides basic exploratory data analysis and sanity checks for the CPUC AV dataset.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Data directory
DATA_DIR = Path('../processed')

## Load Data

In [ ]:
# Load all available datasets
datasets = {}

files = {
    'trips': 'trips.parquet',
    'monthly': 'monthly_summary.parquet',
    'vmt': 'vmt_by_period.parquet',
    'incidents': 'incidents.parquet',
    'tracts': 'tract_pickups.parquet'
}

for name, filename in files.items():
    filepath = DATA_DIR / filename
    if filepath.exists():
        datasets[name] = pd.read_parquet(filepath)
        print(f"Loaded {name}: {len(datasets[name]):,} rows, {len(datasets[name].columns)} columns")
    else:
        print(f"Not found: {filename}")

## Dataset Overview

In [ ]:
# Display basic info for each dataset
for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns: {list(df.columns)}")
    print(f"\nData types:")
    print(df.dtypes)
    print(f"\nNull counts:")
    print(df.isnull().sum())

## Monthly Summary Analysis

In [ ]:
if 'monthly' in datasets:
    monthly = datasets['monthly'].copy()
    
    # Display first few rows
    display(monthly.head(10))
    
    # Summary statistics
    print("\nSummary Statistics:")
    display(monthly.describe())

In [ ]:
if 'monthly' in datasets:
    monthly = datasets['monthly'].copy()
    
    # Time series plots
    if 'reporting_month' in monthly.columns and 'total_trips' in monthly.columns:
        monthly['reporting_month'] = pd.to_datetime(monthly['reporting_month'])
        monthly = monthly.sort_values('reporting_month')
        
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Total trips over time
        ax = axes[0, 0]
        for company in monthly['_company'].unique():
            data = monthly[monthly['_company'] == company]
            ax.plot(data['reporting_month'], data['total_trips'], label=company, marker='o')
        ax.set_title('Total Trips Over Time')
        ax.set_xlabel('Month')
        ax.set_ylabel('Trips')
        ax.legend()
        ax.tick_params(axis='x', rotation=45)
        
        # Total passenger miles
        ax = axes[0, 1]
        if 'total_passenger_miles' in monthly.columns:
            for company in monthly['_company'].unique():
                data = monthly[monthly['_company'] == company]
                ax.plot(data['reporting_month'], data['total_passenger_miles'], label=company, marker='o')
            ax.set_title('Passenger Miles Over Time')
            ax.set_xlabel('Month')
            ax.set_ylabel('Passenger Miles')
            ax.legend()
            ax.tick_params(axis='x', rotation=45)
        
        # VMT over time
        ax = axes[1, 0]
        if 'total_vmt' in monthly.columns:
            for company in monthly['_company'].unique():
                data = monthly[monthly['_company'] == company]
                ax.plot(data['reporting_month'], data['total_vmt'], label=company, marker='o')
            ax.set_title('VMT Over Time')
            ax.set_xlabel('Month')
            ax.set_ylabel('Vehicle Miles Traveled')
            ax.legend()
            ax.tick_params(axis='x', rotation=45)
        
        # Average trip length
        ax = axes[1, 1]
        if 'derived_avg_trip_length' in monthly.columns:
            for company in monthly['_company'].unique():
                data = monthly[monthly['_company'] == company]
                ax.plot(data['reporting_month'], data['derived_avg_trip_length'], label=company, marker='o')
            ax.set_title('Average Trip Length Over Time')
            ax.set_xlabel('Month')
            ax.set_ylabel('Miles per Trip')
            ax.legend()
            ax.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()

## VMT Analysis

In [ ]:
if 'vmt' in datasets:
    vmt = datasets['vmt'].copy()
    
    # Display first few rows
    display(vmt.head(10))
    
    # VMT breakdown by period
    vmt_cols = ['p0_vmt', 'p1_vmt', 'p2_vmt', 'p3_vmt']
    available_cols = [c for c in vmt_cols if c in vmt.columns]
    
    if available_cols:
        totals = vmt[available_cols].sum()
        
        fig, ax = plt.subplots(figsize=(8, 6))
        totals.plot(kind='bar', ax=ax, color=['green', 'orange', 'red', 'gray'][:len(available_cols)])
        ax.set_title('Total VMT by Period')
        ax.set_xlabel('Period')
        ax.set_ylabel('Vehicle Miles Traveled')
        ax.tick_params(axis='x', rotation=45)
        
        # Add percentage labels
        total = totals.sum()
        for i, v in enumerate(totals):
            ax.text(i, v + total*0.01, f'{v/total*100:.1f}%', ha='center')
        
        plt.tight_layout()
        plt.show()

In [ ]:
if 'vmt' in datasets and 'derived_deadhead_pct' in datasets['vmt'].columns:
    vmt = datasets['vmt'].copy()
    
    if 'reporting_period' in vmt.columns:
        vmt['reporting_period'] = pd.to_datetime(vmt['reporting_period'])
        vmt = vmt.sort_values('reporting_period')
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        for company in vmt['_company'].unique():
            data = vmt[vmt['_company'] == company]
            ax.plot(data['reporting_period'], data['derived_deadhead_pct'], label=company, marker='o')
        
        ax.set_title('Deadhead Percentage Over Time')
        ax.set_xlabel('Month')
        ax.set_ylabel('Deadhead %')
        ax.legend()
        ax.tick_params(axis='x', rotation=45)
        ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% line')
        
        plt.tight_layout()
        plt.show()

## Redaction Analysis

In [ ]:
# Analyze redacted values across all datasets
redaction_summary = []

for name, df in datasets.items():
    redacted_cols = [c for c in df.columns if c.endswith('_redacted')]
    
    for col in redacted_cols:
        base_col = col.replace('_redacted', '')
        redacted_count = df[col].sum() if df[col].dtype == bool else 0
        total_count = len(df)
        
        redaction_summary.append({
            'dataset': name,
            'column': base_col,
            'redacted_count': redacted_count,
            'total_count': total_count,
            'redacted_pct': redacted_count / total_count * 100 if total_count > 0 else 0
        })

if redaction_summary:
    redaction_df = pd.DataFrame(redaction_summary)
    redaction_df = redaction_df.sort_values('redacted_pct', ascending=False)
    print("\nRedaction Summary:")
    display(redaction_df)
    
    # Plot
    if len(redaction_df) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        top_redacted = redaction_df.head(15)
        bars = ax.barh(range(len(top_redacted)), top_redacted['redacted_pct'])
        ax.set_yticks(range(len(top_redacted)))
        ax.set_yticklabels([f"{r['dataset']}: {r['column']}" for _, r in top_redacted.iterrows()])
        ax.set_xlabel('Redacted %')
        ax.set_title('Most Redacted Fields')
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()
else:
    print("No redaction flags found in datasets")

## Data Quality Checks

In [ ]:
# Check for temporal gaps
def check_temporal_gaps(df, date_col, company_col='_company'):
    """Check for missing months in time series."""
    if date_col not in df.columns:
        return None
    
    gaps = []
    
    for company in df[company_col].unique():
        company_df = df[df[company_col] == company].copy()
        company_df[date_col] = pd.to_datetime(company_df[date_col])
        dates = company_df[date_col].dropna().sort_values()
        
        if len(dates) < 2:
            continue
        
        expected = pd.date_range(start=dates.min(), end=dates.max(), freq='MS')
        actual = set(dates.dt.to_period('M'))
        missing = set(expected.to_period('M')) - actual
        
        if missing:
            gaps.append({
                'company': company,
                'missing_months': len(missing),
                'examples': sorted(missing)[:5]
            })
    
    return gaps

if 'monthly' in datasets:
    gaps = check_temporal_gaps(datasets['monthly'], 'reporting_month')
    if gaps:
        print("Temporal gaps found in monthly data:")
        for g in gaps:
            print(f"  {g['company']}: {g['missing_months']} missing months")
            print(f"    Examples: {g['examples']}")
    else:
        print("No temporal gaps found in monthly data")

In [ ]:
# Check for outliers
def detect_outliers(df, numeric_cols, threshold=3):
    """Detect values more than threshold std devs from mean."""
    outliers = {}
    
    for col in numeric_cols:
        if col not in df.columns:
            continue
        
        values = df[col].dropna()
        if len(values) < 3:
            continue
        
        mean, std = values.mean(), values.std()
        if std == 0:
            continue
        
        z_scores = (values - mean) / std
        outlier_mask = abs(z_scores) > threshold
        
        if outlier_mask.any():
            outliers[col] = {
                'count': outlier_mask.sum(),
                'values': values[outlier_mask].tolist()[:5]
            }
    
    return outliers

if 'monthly' in datasets:
    numeric_cols = ['total_trips', 'total_passengers', 'total_passenger_miles', 'total_vmt']
    outliers = detect_outliers(datasets['monthly'], numeric_cols)
    
    if outliers:
        print("Outliers detected in monthly data:")
        for col, info in outliers.items():
            print(f"  {col}: {info['count']} outliers")
            print(f"    Example values: {info['values']}")
    else:
        print("No significant outliers detected")

## Company Comparison

In [ ]:
if 'monthly' in datasets:
    monthly = datasets['monthly'].copy()
    
    # Aggregate by company
    company_totals = monthly.groupby('_company').agg({
        'total_trips': 'sum',
        'total_passengers': 'sum',
        'total_passenger_miles': 'sum',
        'total_vmt': 'sum'
    }).round(0)
    
    print("\nTotal by Company:")
    display(company_totals)
    
    # Bar chart comparison
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    metrics = ['total_trips', 'total_passengers', 'total_passenger_miles', 'total_vmt']
    titles = ['Total Trips', 'Total Passengers', 'Passenger Miles', 'VMT']
    
    for ax, metric, title in zip(axes.flat, metrics, titles):
        if metric in company_totals.columns:
            company_totals[metric].plot(kind='bar', ax=ax)
            ax.set_title(title)
            ax.set_xlabel('Company')
            ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## Export Summary Statistics

In [ ]:
# Create summary for documentation
summary = {
    'datasets': {},
    'date_ranges': {},
    'companies': set()
}

for name, df in datasets.items():
    summary['datasets'][name] = {
        'rows': len(df),
        'columns': len(df.columns),
        'null_pct': df.isnull().mean().mean() * 100
    }
    
    if '_company' in df.columns:
        summary['companies'].update(df['_company'].unique())
    
    # Find date range
    date_cols = [c for c in df.columns if 'date' in c.lower() or 'month' in c.lower() or 'period' in c.lower()]
    for col in date_cols:
        try:
            dates = pd.to_datetime(df[col])
            summary['date_ranges'][f"{name}.{col}"] = {
                'min': str(dates.min()),
                'max': str(dates.max())
            }
        except:
            pass

summary['companies'] = list(summary['companies'])

print("\n=== Dataset Summary ===")
print(f"\nCompanies: {summary['companies']}")
print(f"\nDatasets:")
for name, info in summary['datasets'].items():
    print(f"  {name}: {info['rows']:,} rows, {info['columns']} cols, {info['null_pct']:.1f}% null")
print(f"\nDate Ranges:")
for name, range_info in summary['date_ranges'].items():
    print(f"  {name}: {range_info['min']} to {range_info['max']}")